# Brain Tumor MRI Image Classification

Train a custom CNN, MobileNetV2 and EfficientNetB0, compare their measured results,
and download a submission ZIP with models, code, charts and a generated report.

**Before running:** choose **Runtime → Change runtime type → T4 GPU** or another
available GPU. Run the cells from top to bottom. This notebook contains the project
scripts, so you do not need to upload the source ZIP as well.

**Status:** this notebook has not been run on your MRI images. It includes no
trained weights or claimed MRI scores. Training creates those outputs.
This is an educational prototype without clinical validation.


In [ ]:
# 1. Write the project files and install the training dependencies.
from pathlib import Path
import os
import subprocess
import sys

PROJECT = Path('/content/Brain_Tumor_MRI_Submission')
PROJECT.mkdir(parents=True, exist_ok=True)
os.chdir(PROJECT)
PROJECT_FILES = {'brain_tumor_project.py': '"""Train, evaluate, compare and export brain MRI classification models.\n\nRun: python brain_tumor_project.py --data dataset --output artifacts\nResults are calculated from the supplied images; no performance is hardcoded.\n"""\nfrom __future__ import annotations\n\nimport argparse\nimport csv\nimport hashlib\nimport os\nimport platform\nimport shutil\nimport time\nfrom collections import Counter, defaultdict\nfrom datetime import datetime, timezone\nfrom pathlib import Path\n\nos.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")\n\nimport numpy as np\nfrom PIL import Image, ImageOps\nfrom sklearn.metrics import accuracy_score, classification_report, confusion_matrix\nfrom sklearn.metrics import precision_recall_fscore_support\n\nfrom mri_utils import CLASS_NAMES, DATASET_URL, IMAGE_EXTENSIONS\nfrom mri_utils import discover_dataset_root, load_image_array, write_json\n\nMODEL_NAMES = ("custom_cnn", "mobilenet_v2", "efficientnet_b0")\ntf = None  # Imported only when training starts, so data inspection needs no GPU.\n\n\ndef write_csv(path, rows, fields):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    with path.open("w", newline="", encoding="utf-8") as f:\n        writer = csv.DictWriter(f, fieldnames=fields)\n        writer.writeheader()\n        writer.writerows(rows)\n\n\ndef scan_dataset(root):\n    """Validate images and preserve the provided split assignments."""\n    children = {p.name.lower(): p for p in root.iterdir() if p.is_dir()}\n    split_dirs = {\n        "train": children["train"],\n        "validation": next(children[k] for k in ("valid", "validation", "val") if k in children),\n        "test": children["test"],\n    }\n    records, rejected = [], []\n    for split, split_dir in split_dirs.items():\n        folders = {p.name.lower(): p for p in split_dir.iterdir() if p.is_dir()}\n        missing = set(CLASS_NAMES) - set(folders)\n        if missing:\n            raise ValueError(f"{split}: missing class folders {sorted(missing)}")\n        for label, class_name in enumerate(CLASS_NAMES):\n            for path in sorted(folders[class_name].rglob("*")):\n                if not path.is_file() or path.suffix.lower() not in IMAGE_EXTENSIONS:\n                    continue\n                relative = path.relative_to(root).as_posix()\n                try:\n                    with Image.open(path) as image:\n                        image = ImageOps.exif_transpose(image).convert("RGB")\n                        image.load()\n                        width, height = image.size\n                        digest = hashlib.sha256(\n                            f"{width},{height},RGB:".encode() + image.tobytes()\n                        ).hexdigest()\n                    # Roboflow names identify transformed versions of one source.\n                    family = path.name.split(".rf.")[0] if ".rf." in path.name else None\n                    records.append(dict(\n                        path=str(path), relative_path=relative, split=split,\n                        class_name=class_name, label=label, width=width, height=height,\n                        pixel_hash=digest, source_family=family,\n                    ))\n                except (OSError, ValueError, Image.DecompressionBombError) as exc:\n                    rejected.append(dict(path=relative, reason=f"unreadable: {exc}"))\n    if not records:\n        raise ValueError("No readable class-labelled images were found.")\n    return records, rejected\n\n\ndef remove_leakage(records):\n    """Union exact pixel duplicates and named source families, then keep one.\n\n    Priority is test, validation, train. A conflicting-label family is removed\n    entirely. Keeping one member per family also avoids repeated-source weighting.\n    This does not establish patient independence or detect all near duplicates.\n    """\n    parents = list(range(len(records)))\n\n    def find(i):\n        while parents[i] != i:\n            parents[i] = parents[parents[i]]\n            i = parents[i]\n        return i\n\n    def union(i, j):\n        parents[find(i)] = find(j)\n\n    seen_hash, seen_family = {}, {}\n    for i, record in enumerate(records):\n        for value, seen in ((record["pixel_hash"], seen_hash), (record["source_family"], seen_family)):\n            if value is None:\n                continue\n            if value in seen:\n                union(i, seen[value])\n            else:\n                seen[value] = i\n    groups = defaultdict(list)\n    for i, record in enumerate(records):\n        groups[find(i)].append(record)\n    kept, rejected = [], []\n    priority = {"test": 0, "validation": 1, "train": 2}\n    for group in groups.values():\n        if len({r["label"] for r in group}) > 1:\n            rejected.extend(dict(path=r["relative_path"], reason="conflicting labels in source/duplicate family") for r in group)\n            continue\n        group.sort(key=lambda r: (priority[r["split"]], r["relative_path"]))\n        kept.append(group[0])\n        rejected.extend(dict(\n            path=r["relative_path"],\n            reason=f"duplicate/source family; retained {group[0][\'relative_path\']}"\n        ) for r in group[1:])\n    kept.sort(key=lambda r: (r["split"], r["label"], r["relative_path"]))\n    for split in ("train", "validation", "test"):\n        for name in CLASS_NAMES:\n            if not any(r["split"] == split and r["class_name"] == name for r in kept):\n                raise ValueError(f"Cleaning left no {name} images in {split}. Check the source data.")\n    return kept, rejected\n\n\ndef make_dataset(records, size, batch_size, training, seed):\n    paths = [r["path"] for r in records]\n    labels = [r["label"] for r in records]\n    ds = tf.data.Dataset.from_tensor_slices((paths, labels))\n    if training:\n        ds = ds.shuffle(len(records), seed=seed, reshuffle_each_iteration=True)\n\n    def decode(path, label):\n        def read(path_bytes):\n            if isinstance(path_bytes, np.ndarray):\n                path_bytes = path_bytes.item()\n            return load_image_array(path_bytes.decode("utf-8"), size)\n        image = tf.numpy_function(read, [path], tf.float32)\n        image.set_shape((size, size, 3))\n        return image, tf.cast(label, tf.int32)\n\n    ds = ds.map(decode, num_parallel_calls=tf.data.AUTOTUNE).batch(batch_size)\n    if training:\n        augmentation = tf.keras.Sequential([\n            tf.keras.layers.RandomFlip("horizontal", seed=seed),\n            tf.keras.layers.RandomRotation(0.04, fill_mode="reflect", seed=seed + 1),\n            tf.keras.layers.RandomZoom(0.08, fill_mode="reflect", seed=seed + 2),\n            tf.keras.layers.RandomTranslation(0.05, 0.05, fill_mode="reflect", seed=seed + 3),\n            tf.keras.layers.RandomBrightness(0.08, value_range=(0, 255), seed=seed + 4),\n        ], name="training_augmentation")\n        ds = ds.map(lambda x, y: (augmentation(x, training=True), y), num_parallel_calls=tf.data.AUTOTUNE)\n    # No decoded-image cache: prevents large 224x224 RGB datasets filling RAM.\n    return ds.prefetch(tf.data.AUTOTUNE)\n\n\ndef build_model(name, size, weights="imagenet"):\n    layers = tf.keras.layers\n    inputs = layers.Input(shape=(size, size, 3), name="raw_rgb_pixels")\n    base = None\n    if name == "custom_cnn":\n        x = layers.Rescaling(1 / 255.0, name="normalization_0_to_1")(inputs)\n        for index, filters in enumerate((32, 64, 128, 256)):\n            x = layers.Conv2D(filters, 3, padding="same", use_bias=False)(x)\n            x = layers.BatchNormalization()(x)\n            x = layers.Activation("relu")(x)\n            x = layers.MaxPooling2D()(x)\n            if index in (1, 3):\n                x = layers.Dropout(0.15)(x)\n        x = layers.GlobalAveragePooling2D()(x)\n    elif name == "mobilenet_v2":\n        # MobileNetV2 requires [-1,1], embedded in the saved model.\n        x = layers.Rescaling(1 / 127.5, offset=-1.0, name="mobilenet_normalization")(inputs)\n        base = tf.keras.applications.MobileNetV2(\n            include_top=False, weights=weights, input_shape=(size, size, 3)\n        )\n        base.trainable = False\n        x = layers.GlobalAveragePooling2D()(base(x, training=False))\n    elif name == "efficientnet_b0":\n        # EfficientNet already includes its own normalization: input stays 0..255.\n        base = tf.keras.applications.EfficientNetB0(\n            include_top=False, weights=weights, input_shape=(size, size, 3)\n        )\n        base.trainable = False\n        x = layers.GlobalAveragePooling2D()(base(inputs, training=False))\n    else:\n        raise ValueError(f"Unknown model: {name}")\n    x = layers.Dense(128, activation="relu")(x)\n    x = layers.Dropout(0.4)(x)\n    outputs = layers.Dense(len(CLASS_NAMES), activation="softmax", name="class_probabilities")(x)\n    return tf.keras.Model(inputs, outputs, name=name), base\n\n\ndef compile_model(model, learning_rate):\n    model.compile(\n        optimizer=tf.keras.optimizers.Adam(learning_rate),\n        loss="sparse_categorical_crossentropy", metrics=["accuracy"],\n    )\n\n\ndef callbacks(checkpoint):\n    return [\n        tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=4, restore_best_weights=True),\n        tf.keras.callbacks.ModelCheckpoint(str(checkpoint), monitor="val_loss", save_best_only=True),\n        tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2, min_lr=1e-6),\n        tf.keras.callbacks.TerminateOnNaN(),\n    ]\n\n\ndef calculate_metrics(labels, probabilities):\n    predictions = probabilities.argmax(axis=1)\n    precision, recall, f1, _ = precision_recall_fscore_support(\n        labels, predictions, labels=list(range(4)), average="macro", zero_division=0\n    )\n    return dict(\n        accuracy=float(accuracy_score(labels, predictions)),\n        macro_precision=float(precision), macro_recall=float(recall), macro_f1=float(f1),\n        classification_report=classification_report(\n            labels, predictions, labels=list(range(4)), target_names=CLASS_NAMES,\n            output_dict=True, zero_division=0,\n        ),\n        confusion_matrix=confusion_matrix(labels, predictions, labels=list(range(4))).tolist(),\n    )\n\n\ndef plots_for_data(records, output):\n    import matplotlib\n    matplotlib.use("Agg")\n    import matplotlib.pyplot as plt\n    import pandas as pd\n    counts = pd.DataFrame([\n        dict(split=s, class_name=c, images=sum(r["split"] == s and r["class_name"] == c for r in records))\n        for s in ("train", "validation", "test") for c in CLASS_NAMES\n    ])\n    counts.to_csv(output / "class_counts.csv", index=False)\n    ax = counts.pivot(index="class_name", columns="split", values="images").plot.bar(figsize=(9, 5))\n    ax.set_ylabel("Images after duplicate/source cleaning")\n    ax.set_xlabel("Class")\n    ax.set_title("Dataset class distribution")\n    plt.xticks(rotation=15)\n    plt.tight_layout()\n    plt.savefig(output / "class_distribution.png", dpi=170)\n    plt.close()\n    fig, axes = plt.subplots(2, 4, figsize=(12, 6))\n    for column, name in enumerate(CLASS_NAMES):\n        samples = [r for r in records if r["class_name"] == name and r["split"] == "train"][:2]\n        for row in range(2):\n            axes[row, column].axis("off")\n            if row < len(samples):\n                with Image.open(samples[row]["path"]) as image:\n                    axes[row, column].imshow(ImageOps.exif_transpose(image).convert("RGB"))\n                axes[row, column].set_title(name.replace("_", " "))\n    fig.tight_layout()\n    fig.savefig(output / "sample_images.png", dpi=150)\n    plt.close(fig)\n\n\ndef plot_model(history_rows, matrix, name, output):\n    import matplotlib.pyplot as plt\n    import pandas as pd\n    from sklearn.metrics import ConfusionMatrixDisplay\n    frame = pd.DataFrame(history_rows)\n    frame.to_csv(output / f"{name}_history.csv", index=False)\n    fig, axes = plt.subplots(1, 2, figsize=(10, 4))\n    for axis, metric in zip(axes, ("accuracy", "loss")):\n        axis.plot(frame["epoch"], frame[metric], label="Training")\n        axis.plot(frame["epoch"], frame[f"val_{metric}"], label="Validation")\n        axis.set(xlabel="Epoch", ylabel=metric.capitalize(), title=f"{name}: {metric}")\n        axis.legend()\n    fig.tight_layout()\n    fig.savefig(output / f"{name}_training_curves.png", dpi=170)\n    plt.close(fig)\n    fig, axis = plt.subplots(figsize=(7, 6))\n    ConfusionMatrixDisplay(np.array(matrix), display_labels=CLASS_NAMES).plot(\n        ax=axis, cmap="Blues", colorbar=False, xticks_rotation=25,\n    )\n    axis.set_title(f"{name}: held-out test confusion matrix")\n    fig.tight_layout()\n    fig.savefig(output / f"{name}_confusion_matrix.png", dpi=170)\n    plt.close(fig)\n\n\ndef generate_report(output, comparisons, selected, summary):\n    """Write an honest report, using only metrics produced by the current run."""\n    import pandas as pd\n    text = [\n        "# Brain Tumor MRI Image Classification", "",\n        "## Objective", "",\n        "Compare a custom convolutional network with ImageNet transfer learning for "\n        "four MRI image classes and provide a Streamlit research demonstration.", "",\n        "## Dataset and cleaning", "",\n        f"Assigned source: {DATASET_URL}", "",\n        "Classes: glioma, meningioma, no_tumor, pituitary.", "",\n        f"Readable source images: {summary[\'readable_before_cleaning\']}. "\n        f"Retained: {summary[\'retained_images\']}. Excluded: {summary[\'excluded_images\']}.", "",\n        "Exact decoded-pixel duplicates and Roboflow source-name families were grouped. "\n        "One image per family was retained, prioritizing test, validation, then train. "\n        "Conflicting-label families were excluded. See data_manifest.csv and excluded_images.csv.", "",\n        "## Method", "",\n        "Images are decoded as RGB, corrected for EXIF orientation, and resized with PIL bilinear "\n        "interpolation. Training uses mild rotation, zoom, translation, horizontal flipping and "\n        "brightness changes. Validation and test images are never augmented. "\n        "Normalization is embedded in each saved model: custom CNN 0–1, MobileNetV2 −1–1, "\n        "and EfficientNetB0\'s built-in preprocessing on raw 0–255 input.", "",\n        "Balanced class weights, dropout, batch normalization, early stopping, learning-rate "\n        "reduction and best-validation-loss checkpoints are used. The transfer backbones are "\n        "initially frozen; optional final-layer fine-tuning keeps batch normalization frozen.", "",\n        "## Model comparison", "",\n        "| Model | Validation accuracy | Validation macro F1 | Test accuracy | Test macro F1 | Parameters | Inference ms |",\n        "|---|---:|---:|---:|---:|---:|---:|",\n    ]\n    for r in comparisons:\n        text.append(f"| {r[\'model\']} | {r[\'validation_accuracy\']:.4f} | {r[\'validation_macro_f1\']:.4f} | "\n                    f"{r[\'test_accuracy\']:.4f} | {r[\'test_macro_f1\']:.4f} | {r[\'parameters\']:,} | {r[\'inference_ms\']:.2f} |")\n    text += [\n        "", f"Selected deployment model: **{selected}**, chosen by validation macro F1 "\n        "(validation accuracy, then parameter count break ties) before computing test metrics.", "",\n        "Test accuracy, macro precision, macro recall, macro F1, per-class results and confusion "\n        "matrices are stored in the results files. Latency is the mean of ten warmed single-image "\n        "forward passes on the recorded runtime; it is not an end-to-end web latency measurement.", "",\n        "![Model comparison](model_comparison.png)", "",\n        "![Class distribution](class_distribution.png)", "",\n        "## Deployment", "",\n        "Run `streamlit run app.py` from the project folder. The application reads "\n        "artifacts/deployment/model.keras and its metadata, and uses the same RGB and resize code "\n        "as training. It shows all four softmax scores; these are not calibrated clinical probabilities.", "",\n        "## Limitations", "",\n        "This is an image classification educational prototype, not a diagnostic or triage system. "\n        "Patient IDs are not supplied, so patient-level independence cannot be verified. "\n        "The cleaning does not detect every near duplicate. Results on this small curated dataset "\n        "do not establish performance on other hospitals, scanners, populations, MRI sequences, "\n        "or non-MRI uploads. A no_tumor prediction is not evidence that a person is tumor-free. "\n        "Independent patient-level external validation, probability calibration, and clinical "\n        "review would be needed before any medical use.", "",\n        "## Reproducibility", "",\n        "See run_config.json, environment.json, class_names.json, the retained-image manifest, "\n        "training history CSVs, and test_predictions.csv. A fixed seed improves repeatability; "\n        "hardware and library differences can still change training results.", "",\n        "## Dataset attribution", "",\n        "The supplied README identifies Labeled MRI Brain Tumor Dataset Version 1 by the Roboflow "\n        "user Ali Rostami and CC BY 4.0. Original page: "\n        "https://universe.roboflow.com/ali-rostami/labeled-mri-brain-tumor-dataset . "\n        "The provider\'s labeling statements have not been independently verified.", "",\n    ]\n    (output / "REPORT.md").write_text("\\n".join(text), encoding="utf-8")\n    df = pd.DataFrame(comparisons)\n    df.to_csv(output / "model_comparison.csv", index=False)\n    import matplotlib.pyplot as plt\n    ax = df.set_index("model")[["validation_macro_f1", "test_macro_f1"]].plot.bar(figsize=(9, 5), ylim=(0, 1))\n    ax.set(xlabel="Model", ylabel="Macro F1", title="Validation and held-out test comparison")\n    plt.xticks(rotation=15)\n    plt.tight_layout()\n    plt.savefig(output / "model_comparison.png", dpi=170)\n    plt.close()\n\n\ndef train(args):\n    global tf\n    import tensorflow as tensorflow\n    tf = tensorflow\n    tf.keras.utils.set_random_seed(args.seed)\n    try:\n        tf.config.experimental.enable_op_determinism()\n    except (AttributeError, RuntimeError):\n        pass\n    for gpu in tf.config.list_physical_devices("GPU"):\n        try:\n            tf.config.experimental.set_memory_growth(gpu, True)\n        except RuntimeError:\n            pass\n    root = discover_dataset_root(args.data)\n    output = Path(args.output).resolve()\n    # Prevent old results being mixed into a new submission.\n    if output.exists() and any(output.iterdir()):\n        raise ValueError(f"Output directory is not empty: {output}. Use a new --output directory.")\n    models_dir, results_dir = output / "models", output / "results"\n    models_dir.mkdir(parents=True, exist_ok=True)\n    results_dir.mkdir(parents=True, exist_ok=True)\n    scanned, unreadable = scan_dataset(root)\n    records, duplicates = remove_leakage(scanned)\n    excluded = unreadable + duplicates\n    manifest_fields = ["relative_path", "split", "class_name", "label", "width", "height", "pixel_hash", "source_family"]\n    write_csv(results_dir / "data_manifest.csv", [{k:r[k] for k in manifest_fields} for r in records], manifest_fields)\n    write_csv(results_dir / "excluded_images.csv", excluded, ["path", "reason"])\n    summary = dict(readable_before_cleaning=len(scanned), unreadable_images=len(unreadable),\n                   retained_images=len(records), excluded_images=len(excluded))\n    write_json(results_dir / "dataset_summary.json", summary)\n    write_json(output / "class_names.json", CLASS_NAMES)\n    write_json(results_dir / "run_config.json", {**vars(args), "resolved_dataset_root": str(root)})\n    import sklearn\n    import PIL\n    write_json(results_dir / "environment.json", dict(\n        python=platform.python_version(), tensorflow=tf.__version__, numpy=np.__version__,\n        pillow=PIL.__version__, sklearn=sklearn.__version__, platform=platform.platform(),\n        gpu_devices=[d.name for d in tf.config.list_physical_devices("GPU")],\n        generated_utc=datetime.now(timezone.utc).isoformat(),\n    ))\n    plots_for_data(records, results_dir)\n    split_records = {s: [r for r in records if r["split"] == s] for s in ("train", "validation", "test")}\n    print("Retained split counts:", {s:len(v) for s,v in split_records.items()})\n    train_counts = Counter(r["label"] for r in split_records["train"])\n    class_weights = {k:len(split_records["train"]) / (len(CLASS_NAMES) * train_counts[k]) for k in range(4)}\n    write_json(results_dir / "class_weights.json", class_weights)\n    datasets = {s:make_dataset(v, args.image_size, args.batch_size, s == "train", args.seed) for s,v in split_records.items()}\n    validation_labels = np.array([r["label"] for r in split_records["validation"]])\n    test_labels = np.array([r["label"] for r in split_records["test"]])\n    candidates, histories = [], {}\n    weights = "imagenet" if args.weights == "imagenet" else None\n    if weights is None:\n        print("WARNING: weights=none is a software test mode, not pretrained transfer learning.")\n    for name in args.models:\n        tf.keras.backend.clear_session()\n        tf.keras.utils.set_random_seed(args.seed)\n        print(f"\\nTraining {name}")\n        model, base = build_model(name, args.image_size, weights)\n        compile_model(model, 1e-3)\n        stage1_path = models_dir / f"{name}_stage1.keras"\n        started = time.perf_counter()\n        history = model.fit(datasets["train"], validation_data=datasets["validation"], epochs=args.epochs,\n                            class_weight=class_weights, callbacks=callbacks(stage1_path), verbose=2)\n        rows = [dict(epoch=i+1, phase="initial", **{k:float(v[i]) for k,v in history.history.items()})\n                for i in range(len(history.history["loss"]))]\n        model.load_weights(stage1_path)  # Best epoch, not the last epoch.\n        first_loss = float(model.evaluate(datasets["validation"], verbose=0)[0])\n        chosen_path = stage1_path\n        if base is not None and args.fine_tune_epochs > 0:\n            base.trainable = True\n            cutoff = max(0, len(base.layers) - args.unfreeze_layers)\n            for i, layer in enumerate(base.layers):\n                layer.trainable = i >= cutoff and not isinstance(layer, tf.keras.layers.BatchNormalization)\n            compile_model(model, 1e-5)\n            stage2_path = models_dir / f"{name}_stage2.keras"\n            fine = model.fit(datasets["train"], validation_data=datasets["validation"], epochs=args.fine_tune_epochs,\n                             class_weight=class_weights, callbacks=callbacks(stage2_path), verbose=2)\n            model.load_weights(stage2_path)\n            fine_loss = float(model.evaluate(datasets["validation"], verbose=0)[0])\n            if fine_loss < first_loss:\n                chosen_path = stage2_path\n            rows += [dict(epoch=len(rows)+i+1, phase="fine_tune", **{k:float(v[i]) for k,v in fine.history.items()})\n                     for i in range(len(fine.history["loss"]))]\n        model = tf.keras.models.load_model(chosen_path, compile=False)\n        # Preserve the already validated native checkpoint without reserializing.\n        shutil.copy2(chosen_path, models_dir / f"{name}.keras")\n        model.save(models_dir / f"{name}.h5", include_optimizer=False)\n        for temporary in models_dir.glob(f"{name}_stage*.keras"):\n            temporary.unlink()\n        probabilities = model.predict(datasets["validation"], verbose=0)\n        metrics = calculate_metrics(validation_labels, probabilities)\n        write_json(results_dir / f"{name}_validation_metrics.json", metrics)\n        candidates.append(dict(model=name, validation_accuracy=metrics["accuracy"],\n                               validation_macro_f1=metrics["macro_f1"], parameters=int(model.count_params()),\n                               training_seconds=float(time.perf_counter()-started)))\n        histories[name] = rows\n    # Selection is fixed before reading any test prediction or test metric.\n    selected = max(candidates, key=lambda r:(r["validation_macro_f1"], r["validation_accuracy"], -r["parameters"]))["model"]\n    comparisons, prediction_rows = [], []\n    for row in candidates:\n        name = row["model"]\n        tf.keras.backend.clear_session()\n        model = tf.keras.models.load_model(models_dir / f"{name}.keras", compile=False)\n        probabilities = model.predict(datasets["test"], verbose=0)\n        metrics = calculate_metrics(test_labels, probabilities)\n        write_json(results_dir / f"{name}_test_metrics.json", metrics)\n        write_csv(results_dir / f"{name}_confusion_matrix.csv", [\n            dict(actual_class=c, **dict(zip(CLASS_NAMES, metrics["confusion_matrix"][i]))) for i,c in enumerate(CLASS_NAMES)\n        ], ["actual_class"] + CLASS_NAMES)\n        plot_model(histories[name], metrics["confusion_matrix"], name, results_dir)\n        sample = load_image_array(split_records["test"][0]["path"], args.image_size)[None]\n        for _ in range(3):\n            model(sample, training=False).numpy()\n        start = time.perf_counter()\n        for _ in range(10):\n            model(sample, training=False).numpy()\n        latency = (time.perf_counter()-start) * 1000 / 10\n        comparisons.append(dict(**row, test_accuracy=metrics["accuracy"],\n                                test_macro_precision=metrics["macro_precision"],\n                                test_macro_recall=metrics["macro_recall"], test_macro_f1=metrics["macro_f1"],\n                                inference_ms=float(latency), selected=name == selected))\n        for record, scores in zip(split_records["test"], probabilities):\n            prediction_rows.append(dict(model=name, path=record["relative_path"], actual=record["class_name"],\n                                        predicted=CLASS_NAMES[int(scores.argmax())],\n                                        **{f"p_{c}":float(scores[i]) for i,c in enumerate(CLASS_NAMES)}))\n    write_csv(results_dir / "test_predictions.csv", prediction_rows,\n              ["model", "path", "actual", "predicted"] + [f"p_{c}" for c in CLASS_NAMES])\n    write_json(results_dir / "model_comparison.json", comparisons)\n    deployment = output / "deployment"\n    deployment.mkdir(exist_ok=True)\n    shutil.copy2(models_dir / f"{selected}.keras", deployment / "model.keras")\n    write_json(deployment / "metadata.json", dict(\n        model_name=selected, class_names=CLASS_NAMES, image_size=args.image_size,\n        input_range="0..255; normalization is inside the saved model",\n        resize="PIL bilinear with EXIF correction and RGB conversion",\n        selection="highest validation macro F1; validation accuracy and parameter count break ties",\n        pretrained_weights=args.weights, seed=args.seed, clinical_validation=False,\n    ))\n    generate_report(results_dir, comparisons, selected, summary)\n    write_json(output / "COMPLETED.json", dict(selected_model=selected, models=args.models,\n        pretrained_weights=args.weights, software_test_mode=args.weights == "none",\n        retained_images=len(records), completed_utc=datetime.now(timezone.utc).isoformat()))\n    print(f"\\nFinished. Selected model: {selected}. Results: {results_dir}")\n    print("Next: python make_submission.py --artifacts", output)\n\n\ndef parse_args():\n    parser = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)\n    parser.add_argument("--data", default="dataset", help="Extracted assigned dataset directory")\n    parser.add_argument("--output", default="artifacts", help="New or empty output directory")\n    parser.add_argument("--models", nargs="+", choices=MODEL_NAMES, default=list(MODEL_NAMES))\n    parser.add_argument("--epochs", type=int, default=12)\n    parser.add_argument("--fine-tune-epochs", type=int, default=3)\n    parser.add_argument("--unfreeze-layers", type=int, default=30)\n    parser.add_argument("--batch-size", type=int, default=32)\n    parser.add_argument("--image-size", type=int, default=224)\n    parser.add_argument("--seed", type=int, default=42)\n    parser.add_argument("--weights", choices=("imagenet", "none"), default="imagenet",\n                        help="Use imagenet for the assignment; none is only for software tests")\n    args = parser.parse_args()\n    if args.epochs < 1 or args.batch_size < 1 or args.image_size < 32:\n        parser.error("epochs and batch-size must be positive; image-size must be at least 32")\n    if args.fine_tune_epochs < 0 or args.unfreeze_layers < 1:\n        parser.error("fine-tune-epochs must be nonnegative and unfreeze-layers positive")\n    if len(set(args.models)) != len(args.models):\n        parser.error("Do not specify the same model twice")\n    return args\n\n\nif __name__ == "__main__":\n    try:\n        train(parse_args())\n    except (ValueError, FileNotFoundError) as exc:\n        raise SystemExit(f"ERROR: {exc}")\n', 'mri_utils.py': '"""Shared, identical image decoding for training and Streamlit inference."""\nfrom __future__ import annotations\n\nimport json\nimport zipfile\nfrom pathlib import Path\n\nimport numpy as np\nfrom PIL import Image, ImageOps\n\nCLASS_NAMES = ["glioma", "meningioma", "no_tumor", "pituitary"]\nIMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}\nDATASET_URL = "https://drive.google.com/drive/folders/1C9ww4JnZ2sh22I-hbt45OR16o4ljGxju?usp=sharing"\n\n\ndef load_image_array(source, size: int) -> np.ndarray:\n    """RGB, EXIF orientation correction, bilinear resize; return RAW 0..255."""\n    with Image.open(source) as image:\n        image = ImageOps.exif_transpose(image).convert("RGB")\n        image = image.resize((size, size), Image.Resampling.BILINEAR)\n        return np.asarray(image, dtype=np.float32).copy()\n\n\ndef read_json(path):\n    return json.loads(Path(path).read_text(encoding="utf-8"))\n\n\ndef write_json(path, value):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    path.write_text(json.dumps(value, indent=2, allow_nan=False), encoding="utf-8")\n\n\ndef safe_extract_zip(archive, destination):\n    """Extract dataset ZIPs without allowing path traversal or symlink entries."""\n    destination = Path(destination).resolve()\n    destination.mkdir(parents=True, exist_ok=True)\n    with zipfile.ZipFile(archive) as z:\n        for member in z.infolist():\n            target = (destination / member.filename).resolve()\n            if not target.is_relative_to(destination):\n                raise ValueError("The ZIP contains a path outside its destination.")\n            if (member.external_attr >> 16) & 0o170000 == 0o120000:\n                raise ValueError("Symlinks are not supported in dataset ZIPs.")\n        z.extractall(destination)\n\n\ndef discover_dataset_root(path):\n    """Find the assigned train/valid/test dataset, even inside a ZIP wrapper."""\n    path = Path(path).resolve()\n    if not path.is_dir():\n        raise ValueError(f"Dataset directory does not exist: {path}")\n    candidates = [path] + [p for p in path.rglob("*") if p.is_dir()]\n    for candidate in candidates:\n        children = {p.name.lower(): p for p in candidate.iterdir() if p.is_dir()}\n        if "train" in children and "test" in children and any(\n            x in children for x in ("valid", "validation", "val")\n        ):\n            return candidate\n    raise ValueError(\n        "Could not find train/, valid/ (or validation/), and test/ folders. "\n        "Download the complete assigned dataset folder and extract its ZIP."\n    )\n', 'app.py': '"""Streamlit research demonstration: streamlit run app.py"""\nfrom __future__ import annotations\n\nimport io\nimport os\nfrom pathlib import Path\n\nos.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")\n\nimport numpy as np\nimport pandas as pd\nimport streamlit as st\nimport tensorflow as tf\nfrom PIL import Image, ImageOps\n\nfrom mri_utils import CLASS_NAMES, load_image_array, read_json\n\nPROJECT_DIR = Path(__file__).resolve().parent\nARTIFACTS = Path(os.environ.get("MRI_ARTIFACTS_DIR", str(PROJECT_DIR / "artifacts"))).resolve()\nMODEL_PATH = ARTIFACTS / "deployment" / "model.keras"\nMETADATA_PATH = ARTIFACTS / "deployment" / "metadata.json"\n\nst.set_page_config(page_title="Brain MRI Classifier", page_icon="🧠", layout="wide")\nst.title("Brain MRI Image Classifier")\nst.caption("Educational research prototype • Four-class MRI image classification")\nst.warning("This model is not clinically validated. Its predictions must not be used for diagnosis or treatment.")\n\n\n@st.cache_resource\ndef load_assets(model_path, metadata_path, model_mtime, metadata_mtime):\n    metadata = read_json(metadata_path)\n    model = tf.keras.models.load_model(model_path, compile=False)\n    if metadata.get("class_names") != CLASS_NAMES or model.output_shape[-1] != len(CLASS_NAMES):\n        raise ValueError("Model class mapping does not match the four-class project.")\n    size = int(metadata["image_size"])\n    if tuple(model.input_shape[1:]) != (size, size, 3):\n        raise ValueError("Model input size does not match deployment metadata.")\n    return model, metadata\n\n\nif not MODEL_PATH.is_file() or not METADATA_PATH.is_file():\n    st.info("Train the models first, then place the generated artifacts folder beside app.py.")\n    st.code("python brain_tumor_project.py --data dataset --output artifacts")\n    st.stop()\n\ntry:\n    model, metadata = load_assets(str(MODEL_PATH), str(METADATA_PATH),\n                                 MODEL_PATH.stat().st_mtime_ns, METADATA_PATH.stat().st_mtime_ns)\nexcept (OSError, ValueError, KeyError) as exc:\n    st.error(f"Could not load the trained model: {exc}")\n    st.stop()\n\nst.sidebar.header("Selected model")\nst.sidebar.write(metadata["model_name"].replace("_", " ").title())\nst.sidebar.caption("Selected using validation macro F1, before test evaluation.")\nst.sidebar.write(f"Input size: {metadata[\'image_size\']} × {metadata[\'image_size\']}")\nst.sidebar.caption("Only JPG or PNG brain MRI images are supported. DICOM is not supported.")\n\nprediction_tab, results_tab = st.tabs(["Classify an MRI image", "Evaluation results"])\nwith prediction_tab:\n    uploaded = st.file_uploader("Upload a brain MRI image", type=["jpg", "jpeg", "png"])\n    st.caption("The classifier cannot verify that an upload is an MRI or reject all unfamiliar images.")\n    if uploaded is not None:\n        if uploaded.size > 20 * 1024 * 1024:\n            st.error("Please upload an image smaller than 20 MB.")\n        else:\n            try:\n                raw = uploaded.getvalue()\n                with Image.open(io.BytesIO(raw)) as opened:\n                    display_image = ImageOps.exif_transpose(opened).convert("RGB").copy()\n                image = load_image_array(io.BytesIO(raw), int(metadata["image_size"]))\n                left, right = st.columns([1, 1])\n                with left:\n                    st.image(display_image, caption="Uploaded MRI image", width="stretch")\n                with right:\n                    if st.button("Classify image", type="primary"):\n                        with st.spinner("Running the model..."):\n                            scores = np.asarray(model(image[None], training=False))[0]\n                        index = int(scores.argmax())\n                        predicted = metadata["class_names"][index].replace("_", " ").title()\n                        st.subheader(f"Predicted image class: {predicted}")\n                        st.metric("Highest model score", f"{scores[index]:.1%}")\n                        values = pd.DataFrame({"Class": [c.replace("_", " ").title() for c in CLASS_NAMES],\n                                               "Model score": scores.astype(float)})\n                        st.bar_chart(values.set_index("Class"))\n                        st.dataframe(values.assign(**{"Model score": values["Model score"].map(lambda x:f"{x:.2%}")}),\n                                     hide_index=True, width="stretch")\n                        st.caption("Scores are softmax outputs and are not calibrated clinical probabilities. "\n                                   "A No Tumor prediction does not establish that a person is tumor-free.")\n            except (OSError, ValueError, Image.DecompressionBombError) as exc:\n                st.error(f"The image could not be processed: {exc}")\n\nwith results_tab:\n    comparison = ARTIFACTS / "results" / "model_comparison.csv"\n    if comparison.is_file():\n        st.dataframe(pd.read_csv(comparison), hide_index=True, width="stretch")\n        for filename, caption in [\n            ("model_comparison.png", "Model comparison"),\n            (f"{metadata[\'model_name\']}_confusion_matrix.png", "Selected model test confusion matrix"),\n            (f"{metadata[\'model_name\']}_training_curves.png", "Selected model training history"),\n        ]:\n            path = ARTIFACTS / "results" / filename\n            if path.is_file():\n                st.image(str(path), caption=caption)\n    else:\n        st.info("Evaluation files were not included in this deployment.")\n', 'make_submission.py': '"""Package verified training outputs and code for upload after a real run."""\nfrom __future__ import annotations\n\nimport argparse\nimport json\nimport zipfile\nfrom pathlib import Path\n\n\ndef package(artifacts, output):\n    project = Path(__file__).resolve().parent\n    artifacts = Path(artifacts).resolve()\n    completed = artifacts / "COMPLETED.json"\n    if not completed.is_file():\n        raise ValueError("Training is not complete: COMPLETED.json is missing.")\n    status = json.loads(completed.read_text())\n    if status.get("software_test_mode") or status.get("pretrained_weights") != "imagenet":\n        raise ValueError("Random-weight software tests cannot be packaged as an assignment result. Run with --weights imagenet.")\n    expected = {"custom_cnn", "mobilenet_v2", "efficientnet_b0"}\n    if set(status["models"]) != expected:\n        raise ValueError("The full submission needs the custom CNN and both transfer models. Train all three.")\n    required = [artifacts / "deployment" / "model.keras", artifacts / "deployment" / "metadata.json",\n                artifacts / "results" / "REPORT.md", artifacts / "results" / "model_comparison.csv",\n                artifacts / "class_names.json"]\n    required += [artifacts / "models" / f"{name}.h5" for name in sorted(expected)]\n    required += [artifacts / "results" / f"{name}_test_metrics.json" for name in sorted(expected)]\n    if any(not p.is_file() for p in required):\n        raise ValueError("Some required training/model/evaluation files are missing.")\n    output = Path(output).resolve()\n    output.parent.mkdir(parents=True, exist_ok=True)\n    with zipfile.ZipFile(output, "w", zipfile.ZIP_DEFLATED, compresslevel=6) as archive:\n        for path in sorted(project.iterdir()):\n            if path.is_file() and path.suffix in {".py", ".md", ".txt", ".ipynb", ".json"}:\n                archive.write(path, Path("Brain_Tumor_MRI_Submission") / path.name)\n        ignore = {".keras"}  # Avoid storing each model twice; selected deployment stays.\n        for path in sorted(artifacts.rglob("*")):\n            if not path.is_file():\n                continue\n            relative = path.relative_to(artifacts)\n            if relative.parts[0] == "models" and path.suffix in ignore:\n                continue\n            archive.write(path, Path("Brain_Tumor_MRI_Submission") / "artifacts" / relative)\n    return output\n\n\nif __name__ == "__main__":\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--artifacts", default="artifacts")\n    parser.add_argument("--output", default="Brain_Tumor_MRI_Trained_Submission.zip")\n    args = parser.parse_args()\n    try:\n        print("Submission ZIP created:", package(args.artifacts, args.output))\n    except ValueError as exc:\n        raise SystemExit(f"ERROR: {exc}")\n', 'requirements.txt': 'tensorflow>=2.19,<2.20\nnumpy>=1.26,<2.2\npandas>=2.2,<3\nscikit-learn>=1.5,<2\nmatplotlib>=3.8,<4\nPillow>=10.4,<12\nstreamlit>=1.50,<1.51\n', 'requirements-colab.txt': "# Keep Colab's TensorFlow if it is already >=2.19 and <2.20.\ntensorflow>=2.19,<2.20\nnumpy>=1.26,<2.2\npandas>=2.2,<3\nscikit-learn>=1.5,<2\nmatplotlib>=3.8,<4\nPillow>=10.4,<12\n", 'README.md': "# Brain Tumor MRI Image Classification\n\nThis project trains a custom CNN, MobileNetV2 and EfficientNetB0 on the assigned\nfour-class MRI dataset, compares them, and deploys the validation-selected model\nin Streamlit. Labels are glioma, meningioma, no_tumor and pituitary.\n\n**Package status:** source code and the Colab notebook are supplied. The real MRI\ndataset and trained weights are not included in the source package. Run training\nbefore submitting a completed project. The scripts compute metrics from your\nrun; no accuracy figures or trained-model claims are prefilled.\n\n## Fastest route for today's submission\n\n1. Open `Brain_Tumor_MRI_Colab.ipynb` in Google Colab using **File → Upload notebook**.\n2. Select **Runtime → Change runtime type → T4 GPU** (or another available GPU).\n3. Download the complete dataset folder from the assignment link as a ZIP:\n   https://drive.google.com/drive/folders/1C9ww4JnZ2sh22I-hbt45OR16o4ljGxju?usp=sharing\n4. Run the notebook cells in order. When asked for a file, upload the **dataset ZIP**.\n   An optional Drive shortcut route is included for larger files.\n5. The final cell downloads `Brain_Tumor_MRI_Trained_Submission.zip` containing\n   code, three `.h5` trained models, the selected Streamlit model, metadata,\n   computed metrics, figures and `REPORT.md`.\n6. Download the executed notebook through **File → Download → Download .ipynb**\n   and submit it alongside the trained ZIP if notebooks are requested.\n7. Upload the source and generated results to your own public GitHub repository\n   and include its link, as required by the assignment.\n\nTraining time depends on GPU availability and the selected epoch counts.\nThe default run uses 12 initial epochs and up to 3 fine-tuning epochs per\ntransfer model. Early stopping can finish sooner. It does not guarantee a score.\nYou can use 8 initial epochs and 2 fine-tuning epochs when time is limited;\nthe quality of that run must still be judged from its actual validation results.\n\n## Files in this package\n\n| File | Purpose |\n|---|---|\n| `brain_tumor_project.py` | Train, clean data, evaluate, compare and export all models |\n| `mri_utils.py` | Shared image decoding, class mapping and safe ZIP extraction |\n| `app.py` | Streamlit image upload, class scores and evaluation display |\n| `make_submission.py` | Validate required outputs and build the trained submission ZIP |\n| `Brain_Tumor_MRI_Colab.ipynb` | GPU notebook with embedded project scripts |\n| `requirements.txt` | Local project dependencies |\n| `requirements-colab.txt` | Training dependencies for Colab |\n| `DATASET_INFO.json` | Verified source split/class counts before cleaning |\n| `SUBMISSION_CHECKLIST.md` | Required deliverables and submission steps |\n\n## Dataset layout\n\nExtract the entire assigned folder, preserving these directories:\n\n- `dataset/train/glioma`, `meningioma`, `no_tumor`, `pituitary`\n- `dataset/valid/glioma`, `meningioma`, `no_tumor`, `pituitary`\n- `dataset/test/glioma`, `meningioma`, `no_tumor`, `pituitary`\n\nEach split has the four class folders. `validation` and `val` are accepted\nalternatives to `valid`. Extra ZIP wrapper directories are detected automatically.\nThe supplied `_classes.csv` files are not required by the loader because the\nclass folder already supplies each label. DICOM files are not supported.\n\n## Local training\n\nUse Python 3.10–3.12 with a compatible TensorFlow installation. A Colab GPU is\nrecommended for completing all three models. CPU training on a laptop can take\nsubstantially longer. From the extracted project directory:\n\n```bash\npython -m venv .venv\nsource .venv/bin/activate\npython -m pip install -r requirements.txt\npython brain_tumor_project.py --data dataset --output artifacts\npython make_submission.py --artifacts artifacts\nstreamlit run app.py\n```\n\nOn Windows, activate the environment using `.venv\\Scripts\\activate`.\n\nFor a shorter real run:\n\n```bash\npython brain_tumor_project.py --data dataset --output artifacts --epochs 8 --fine-tune-epochs 2\n```\n\nIf GPU memory is insufficient, add `--batch-size 16`. A new training run must use\na new or empty output directory. This protects against mixing old and new metrics.\nThe default 224-pixel images are appropriate for the pretrained backbones.\nInternet access is needed for the first download of ImageNet weights.\n\n## Data integrity and evaluation\n\nThe assignment's CSV inventories report 1,695 training, 502 validation and 246\ntest images. Their class counts are recorded in `DATASET_INFO.json`.\nSource prefixes before `.rf.` overlap across the splits: 97 distinct prefixes\nbetween training and validation, 57 between training and test, and 10 between\nvalidation and test. These three overlap counts must not be added because a\nfamily can appear in all three splits.\n\nThe pipeline groups exact decoded-pixel duplicates and Roboflow source-name\nfamilies. It retains one example per family, prioritizing test, then validation,\nthen training. Conflicting-label families and unreadable files are excluded and\nlogged. The retained-image manifest records the actual image dimensions and\nlabels. Cleaned counts differ from the original counts. Files in the source\ndataset are never modified or deleted.\n\nOnly training images are augmented. Class weights are calculated from training\nlabels only. The selected model has the highest validation macro F1; validation\naccuracy and fewer parameters break ties. Selection happens before test\npredictions are computed. All three models are then evaluated on the held-out\ntest split for comparison. Test results must not be used for repeated tuning.\n\nNormalization is part of the saved model, avoiding training/deployment mismatch:\n\n| Model | Input to the saved model | Internal preprocessing |\n|---|---|---|\n| Custom CNN | RGB pixels 0–255 | Scale to 0–1 |\n| MobileNetV2 | RGB pixels 0–255 | Scale to −1–1 |\n| EfficientNetB0 | RGB pixels 0–255 | Built-in EfficientNet preprocessing |\n\nTraining and Streamlit both use EXIF-corrected RGB decoding and PIL bilinear\nresizing. Transfer learning starts with ImageNet weights and a frozen backbone.\nOptional fine-tuning unfreezes the last layers while keeping batch normalization\nfrozen and using a learning rate of 1e-5.\n\n## Generated outputs\n\n- `artifacts/models/`: `.h5` and native `.keras` exports for each trained model.\n- `artifacts/deployment/`: selected `model.keras` and `metadata.json`.\n- `artifacts/results/`: class distribution and sample images, training curves,\n  history CSVs, per-model validation and test metrics, confusion matrices,\n  test predictions, model comparison CSV/JSON/PNG, and `REPORT.md`.\n- `artifacts/class_names.json`: fixed output-label order.\n- `artifacts/COMPLETED.json`: completion status for the packaging check.\n\nThe trained ZIP includes the three required `.h5` exports plus the selected\nnative `.keras` deployment model. It omits redundant per-model `.keras` copies\nand the MRI dataset to keep the upload smaller. TensorFlow considers HDF5 a\nlegacy format; the project includes it because the brief explicitly requests it.\n\n## Run the application after Colab training\n\nExtract the **trained** ZIP on your computer. Its project directory contains\n`app.py` and `artifacts/` together. Install `requirements.txt` and run:\n\n```bash\nstreamlit run app.py\n```\n\nOpen the local URL printed by Streamlit, normally `http://localhost:8501`.\nThe app displays a training instruction when trained weights are absent.\nTo use a different artifacts directory, set the `MRI_ARTIFACTS_DIR` environment\nvariable to its absolute path before starting Streamlit.\n\n## GitHub and submission\n\nCreate a public GitHub repository using your own account. Upload these source\nfiles and the notebook, plus useful results such as comparison figures and\nmetrics. Follow the dataset's licensing terms before redistributing any images.\nThe supplied `.gitignore` keeps model weights and dataset images out of an\nordinary Git commit. Upload the trained ZIP to the assignment portal; for model\nfiles on GitHub use a release asset or Git LFS if repository limits require it.\nNo repository has been created or published by this package.\n\n## Scope and limitations\n\nThis is an educational MRI image classifier. It has not been clinically\nvalidated and cannot diagnose a patient, establish that a person is tumor-free,\nor reliably reject non-MRI/out-of-distribution uploads. The displayed softmax\nscores are not calibrated clinical probabilities. There are no supplied patient\nIDs, so patient-level independence cannot be verified; the grouping method does\nnot detect every near duplicate. Performance on this curated dataset does not\nestablish performance at another hospital or with a different MRI protocol.\n\n`--weights none` is reserved for software verification with synthetic data.\nIt does not implement pretrained transfer learning. `make_submission.py` refuses\nto package such runs as final assignment results.\n\n## Dataset attribution and references\n\nThe dataset README identifies **Labeled MRI Brain Tumor Dataset Version 1**,\nRoboflow user Ali Rostami, and **CC BY 4.0**. Its labeling claims have not been\nindependently verified. Cleaning, preprocessing, and source grouping described\nabove are transformations performed by this project.\n\n- Original dataset page: https://universe.roboflow.com/ali-rostami/labeled-mri-brain-tumor-dataset\n- Dataset README: https://drive.google.com/file/d/1H8DhtZjx4bilk-6kebL6087Xu0TVxv33/view?usp=drivesdk\n- Keras transfer-learning guide: https://keras.io/guides/transfer_learning/\n- Keras MobileNet models: https://keras.io/api/applications/mobilenet/mobilenet_models/\n- Keras EfficientNet models: https://keras.io/api/applications/efficientnet/efficientnet_models/\n- Streamlit resource caching: https://docs.streamlit.io/develop/api-reference/caching-and-state/st.cache_resource\n\n## Software verification\n\nThe delivered code was checked with Python 3.12, TensorFlow CPU 2.19.1 and\nStreamlit 1.50.0. A short synthetic-image run exercised all three models,\nfine-tuning, evaluation, figures, native checkpoints, HDF5 exports and selected\ndeployment creation. Native/HDF5 predictions matched for each model. Additional\nchecks covered duplicate/source-family isolation, conflicting labels, ZIP path\ntraversal rejection, notebook syntax and embedded-source consistency, and app\nstartup both with and without generated model files. The packager rejected\nrandom-weight software-test results.\n\nThese checks verify software behavior, not MRI classification performance.\nNo synthetic models, synthetic metrics or synthetic sample images are included\nin this source package. Full MRI training, GPU execution and clinical validation\nhave not been performed here.\n", 'SUBMISSION_CHECKLIST.md': "# Submission checklist\n\n## Run today\n\n1. Upload `Brain_Tumor_MRI_Colab.ipynb` to Colab and select a GPU runtime.\n2. Run setup and upload the complete assigned dataset ZIP when asked.\n3. Run the full three-model training cell with ImageNet weights.\n4. Confirm that all three comparison rows and their computed test results appear.\n5. Run the export cell and download `Brain_Tumor_MRI_Trained_Submission.zip`.\n6. Download the executed `.ipynb` separately from Colab's File menu.\n7. Publish your code to a public GitHub repository and copy its URL.\n8. Test the Streamlit application locally after extracting the trained ZIP.\n\n## Upload to the assignment portal\n\n- Trained submission ZIP containing scripts, trained `.h5` models, selected\n  deployment model, metadata, actual metrics/charts and generated `REPORT.md`.\n- Executed notebook, if the portal requests a separate notebook file.\n- Your public GitHub repository URL, as required by the project brief.\n- Screenshots or a video only if your evaluator separately requires them.\n\nThe provided source ZIP alone does not contain trained models or measured MRI\nperformance. Do not submit random-weight software-test outputs as final results.\nThere is no supplied test score until the real-data notebook has finished.\n\n## Check the trained archive\n\n- `artifacts/models/custom_cnn.h5`\n- `artifacts/models/mobilenet_v2.h5`\n- `artifacts/models/efficientnet_b0.h5`\n- `artifacts/deployment/model.keras`\n- `artifacts/deployment/metadata.json`\n- `artifacts/results/model_comparison.csv`\n- `artifacts/results/REPORT.md`\n- Per-model test metrics, confusion matrices and training curves.\n- `artifacts/COMPLETED.json`\n\nKeep the test set for final evaluation. The app and report must identify this as\nan educational prototype without clinical validation.\n", 'DATASET_INFO.json': '{\n  "dataset": "Labeled MRI Brain Tumor Dataset Version 1",\n  "assigned_folder_url": "https://drive.google.com/drive/folders/1C9ww4JnZ2sh22I-hbt45OR16o4ljGxju?usp=sharing",\n  "inventory_read_date": "2026-10-12",\n  "attribution": "Dataset README identifies Roboflow user Ali Rostami and CC BY 4.0; labeling claims not independently verified.",\n  "train": {\n    "images": 1695,\n    "counts": {\n      "glioma": 564,\n      "meningioma": 358,\n      "no_tumor": 335,\n      "pituitary": 438\n    }\n  },\n  "validation": {\n    "images": 502,\n    "counts": {\n      "glioma": 161,\n      "meningioma": 124,\n      "no_tumor": 99,\n      "pituitary": 118\n    }\n  },\n  "test": {\n    "images": 246,\n    "counts": {\n      "glioma": 80,\n      "meningioma": 63,\n      "no_tumor": 49,\n      "pituitary": 54\n    }\n  },\n  "train_validation_source_name_overlap": 97,\n  "train_test_source_name_overlap": 57,\n  "validation_test_source_name_overlap": 10,\n  "note": "Counts are original CSV inventory counts. Training removes pixel duplicates and source families; actual retained counts are computed during your run. Source overlap counts are pairwise and must not be added."\n}\n', '.gitignore': '__pycache__/\n.venv/\n.DS_Store\ndataset/\n*.zip\nartifacts/models/\nartifacts/deployment/model.keras\nartifacts/results/sample_images.png\n'}
for filename, contents in PROJECT_FILES.items():
    (PROJECT / filename).write_text(contents, encoding='utf-8')
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements-colab.txt'])
if str(PROJECT) not in sys.path:
    sys.path.insert(0, str(PROJECT))
print('Project files ready:', PROJECT)


## 2. Supply the assigned dataset

Download the complete folder from
[the assignment dataset](https://drive.google.com/drive/folders/1C9ww4JnZ2sh22I-hbt45OR16o4ljGxju?usp=sharing)
as a ZIP. In the next cell, upload the **dataset ZIP**, not the code ZIP.
The root must contain `train`, `valid` and `test`, each with `glioma`,
`meningioma`, `no_tumor` and `pituitary` subfolders.

Alternatively, add a shortcut to the dataset folder in **My Drive**, set
`DATA_SOURCE = 'drive'` and edit `DRIVE_DATASET_DIR` to that folder's mounted path.
The notebook then copies just the discovered dataset root to local Colab storage.

The original inventories list 1,695 training, 502 validation and 246 test images.
Repeated source families cross those splits; the training script removes them
and reports its actual retained counts. The source files are left unchanged.


In [ ]:
# 2. Upload the dataset ZIP, or use a mounted Drive shortcut.
from google.colab import files, drive
import shutil
from mri_utils import safe_extract_zip, discover_dataset_root

DATA_SOURCE = 'zip_upload'  # Change to 'drive' only if you prefer a Drive shortcut.
DRIVE_DATASET_DIR = '/content/drive/MyDrive/Brain_Tumor_MRI_Dataset'
DATA_DIR = Path('/content/mri_dataset')

if DATA_SOURCE == 'zip_upload':
    uploaded = files.upload()
    archive_names = [name for name in uploaded if name.lower().endswith('.zip')]
    if not archive_names:
        raise ValueError('Upload the complete dataset ZIP downloaded from the assignment link.')
    # This directory contains only temporary dataset copies made by this notebook.
    if DATA_DIR.exists():
        shutil.rmtree(DATA_DIR)
    DATA_DIR.mkdir(parents=True)
    for name in archive_names:
        safe_extract_zip(name, DATA_DIR)
    del uploaded  # Release the upload byte buffers before GPU training.
elif DATA_SOURCE == 'drive':
    drive.mount('/content/drive')
    source_root = discover_dataset_root(DRIVE_DATASET_DIR)
    if DATA_DIR.exists():
        shutil.rmtree(DATA_DIR)
    shutil.copytree(source_root, DATA_DIR)
else:
    raise ValueError("DATA_SOURCE must be 'zip_upload' or 'drive'.")

DATA_ROOT = discover_dataset_root(DATA_DIR)
print('Dataset ready:', DATA_ROOT)


## 3. Train all three models

Keep `weights=imagenet` for a real submission. The default is 12 initial epochs
and up to 3 fine-tuning epochs; early stopping may finish a model sooner.
For a shorter run you may choose 8 and 2, respectively, and judge its actual
validation results. GPU speed and availability determine completion time.

The selected deployment model is chosen from validation macro F1 **before**
any test predictions are computed. Test accuracy is not used for selection.
An existing nonempty `artifacts` directory is rejected to avoid mixing runs.


In [ ]:
# 3. Real MRI training. Outputs remain available in the Colab Files panel.
import tensorflow as tf

EPOCHS = 12
FINE_TUNE_EPOCHS = 3
BATCH_SIZE = 32  # Use 16 if your GPU runs out of memory.
ALLOW_CPU = False  # CPU training can take much longer; enable only deliberately.

gpus = tf.config.list_physical_devices('GPU')
print('TensorFlow:', tf.__version__, 'GPU devices:', gpus)
if not gpus and not ALLOW_CPU:
    raise RuntimeError('Select a GPU in Runtime → Change runtime type, then rerun the notebook.')

command = [sys.executable, 'brain_tumor_project.py',
           '--data', str(DATA_ROOT), '--output', 'artifacts',
           '--epochs', str(EPOCHS), '--fine-tune-epochs', str(FINE_TUNE_EPOCHS),
           '--batch-size', str(BATCH_SIZE), '--image-size', '224', '--weights', 'imagenet']
subprocess.check_call(command)
print('Real-data training completed.')


## 4. Inspect the measured results

The following cells show actual retained class counts, validation/test comparisons,
per-class test precision/recall/F1, confusion matrices and training curves.
The generated `REPORT.md` includes the measured comparison and limitations.


In [ ]:
# 4. Read the results from this run.
import json
import pandas as pd
from IPython.display import display, Image as DisplayImage, Markdown

RESULTS = PROJECT / 'artifacts' / 'results'
display(pd.read_csv(RESULTS / 'class_counts.csv'))
comparison = pd.read_csv(RESULTS / 'model_comparison.csv')
display(comparison)
display(DisplayImage(filename=str(RESULTS / 'model_comparison.png')))
display(DisplayImage(filename=str(RESULTS / 'class_distribution.png')))
for model_name in ['custom_cnn', 'mobilenet_v2', 'efficientnet_b0']:
    metrics = json.loads((RESULTS / f'{model_name}_test_metrics.json').read_text())
    print()
    print('Model:', model_name)
    print('Test accuracy:', round(metrics['accuracy'], 4),
          'Macro F1:', round(metrics['macro_f1'], 4))
    report = pd.DataFrame({name:values for name, values in metrics['classification_report'].items()
                           if isinstance(values, dict)}).transpose()
    display(report)
    display(DisplayImage(filename=str(RESULTS / f'{model_name}_confusion_matrix.png')))
    display(DisplayImage(filename=str(RESULTS / f'{model_name}_training_curves.png')))
display(Markdown((RESULTS / 'REPORT.md').read_text(encoding='utf-8')))


## 5. Download the trained submission

The ZIP will contain the project code, the three required `.h5` models, selected
native `.keras` deployment model, metadata, measured results, charts and report.
Dataset images and redundant native model copies are excluded to reduce size.

Also download your **executed notebook** through **File → Download → Download .ipynb**.
Publish the code to your own public GitHub repository and include its link.
The notebook does not create or publish a repository automatically.


In [ ]:
# 5. Validate, build and download the real trained-model submission ZIP.
subprocess.check_call([sys.executable, 'make_submission.py', '--artifacts', 'artifacts',
                       '--output', '/content/Brain_Tumor_MRI_Trained_Submission.zip'])
files.download('/content/Brain_Tumor_MRI_Trained_Submission.zip')


## 6. Run Streamlit on your computer

Extract the trained ZIP, open its project directory in a terminal, and run:

```bash
python -m pip install -r requirements.txt
streamlit run app.py
```

Open the local URL printed by Streamlit. Upload a brain MRI JPG/PNG to show the
predicted image class and all model scores, and open the Evaluation results tab.
The app is an educational research demo, not a diagnosis or treatment tool.
